# Exploratory Data Analysis 

__In this part of the project I will present you the Exploratory Data Analsis for the DataFrame found on this <a href= https://www.kaggle.com/datasets/imakash3011/customer-personality-analysis>link</a>. The main goal of this analysis is to provide insights for business decisions.__

__After the Explorary Data Analysis, I will apply the K-Means technique in order to divide the customers database in cluster for more effective Marketing actions.__

__The libraries used for this project are listed below__

In [ ]:
import sys
sys.path.append('../src')
from custom_functions import error_handling

In [ ]:
try:
    import pandas as pd 
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns 
    from datetime import datetime as dt
    import plotly.express as px 
    import warnings
    warnings.filterwarnings("ignore")
    print('Libraries imported')
except Exception as e:
    error_handling(e)

# Customer Personality 

is a detailed analysis of a company’s ideal customers. It helps a business to better understand its customers and makes it easier for them to modify products according to the specific needs, behaviors and concerns of different types of customers.

Customer personality analysis helps a business to modify its product based on its target customers from different types of customer segments. For example, instead of spending money to market a new product to every customer in the company’s database, a company can analyze which customer segment is most likely to buy the product and then market the product only on that particular segment.

__People__

<code>ID</code>: Customer's unique identifier -
<code>Year_Birth</code>: Customer's birth year - 
<code>Education</code>: Customer's education level - 
<code>Marital_Status</code>: Customer's marital status - 
<code>Income</code>: Customer's yearly household income - 
<code>Kidhome</code>: Number of children in customer's household - 
<code>Teenhome</code>: Number of teenagers in customer's household - 
<code>Dt_Customer</code>: Date of customer's enrollment with the company - 
<code>Recency</code>: Number of days since customer's last purchase - 
<code>Complain</code>: 1 if the customer complained in the last 2 years, 0 otherwise - 

__Products__

<code>MntWines</code>: Amount spent on wine in last 2 years - 
<code>MntFruits</code>: Amount spent on fruits in last 2 years - 
<code>MntMeatProducts</code>: Amount spent on meat in last 2 years - 
<code>MntFishProducts</code>: Amount spent on fish in last 2 years - 
<code>MntSweetProducts</code>: Amount spent on sweets in last 2 years - 
<code>MntGoldProds</code>: Amount spent on gold in last 2 years - 

__Promotion__

<code>NumDealsPurchases</code>: Number of purchases made with a discount - 
<code>AcceptedCmp1</code>: 1 if customer accepted the offer in the 1st campaign, 0 otherwise - 
<code>AcceptedCmp2</code>: 1 if customer accepted the offer in the 2nd campaign, 0 otherwise -
<code>AcceptedCmp3</code>: 1 if customer accepted the offer in the 3rd campaign, 0 otherwise -
<code>AcceptedCmp4</code>: 1 if customer accepted the offer in the 4th campaign, 0 otherwise -
<code>AcceptedCmp5</code>: 1 if customer accepted the offer in the 5th campaign, 0 otherwise - 
<code>Response</code>: 1 if customer accepted the offer in the last campaign, 0 otherwise - 

__Place__

<code>NumWebPurchases</code>: Number of purchases made through the company’s website - 
<code>NumCatalogPurchases</code>: Number of purchases made using a catalogue - 
<code>NumStorePurchases</code>: Number of purchases made directly in stores - 
<code>NumWebVisitsMonth</code>: Number of visits to company’s website in the last month - 


__We will start by loading the .CSV file, creating a DataFrame and printing the number of rows and columns__

In [ ]:
file = '../data/marketing_campaign.csv'
data = pd.read_csv(file, sep= "\t")
df = data.copy()
print(f"Number of rows: {df.shape[0]}\nNumber of columns: {df.shape[1]}")
pd.set_option('display.max_columns', len(df))
display(df.head())

Instead of `df.info()` we will iterate over the columns to describe them in more friendly way, looking at the number of unique values, data type and number of nulls.

In [ ]:
null_cols = []
for col in df.columns:
    dtype = df[col].dtype
    nunique= df[col].nunique()
    nulls = df[col].isnull().sum()
    if nulls > 0:
        null_cols.append(col)
    print(f'{col:<20} | Type: {str(dtype): <8} | Unique: {nunique: <10} | Nulls: {nulls}')

In [ ]:
print('Columns with null values:', *null_cols)

`'Income'` is the only feature with `null` values. Also, it's a numeric (`float64`) one. This issue will be addressed as the EDA progresses, in order to make more informed decisisions

__The columns `Z_CostContact` and `Z_Revenue` were excluded from the dataset as they are constant values with zero variance, providing no information gain or predictive utility for the analysis.__

In [ ]:
constants = ['Z_CostContact', 'Z_Revenue']
df.drop(constants, axis = 1, inplace = True)
print(f'{constants} dropped') if len(np.intersect1d(constants, df.columns)) == 0 else print(f'{constants} not dropped')

To better understand customer purchasing habits, a `Total_Spent` feature should be engineered by aggregating the total monetary expenditure across all product categories — including wines, fruits, meat, fish, sweets, and gold — representing each customer's total investment over the last two years.

In [ ]:
df['Total_Spent'] = df[['MntWines', 'MntFruits',
       'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts',
       'MntGoldProds']].sum(axis=1)
df['Total_Spent'].head()

In [ ]:
df['Total_Purchases'] = df[['NumDealsPurchases', 'NumWebPurchases','NumCatalogPurchases', 'NumStorePurchases']].sum(axis=1)
df['Total_Purchases'].head()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10,6))
ax1.boxplot(df['Total_Purchases'])
ax1.set_title('Total Purchases')
ax2.boxplot(df['Total_Spent'])
ax2.set_title('Total_Spent')
plt.suptitle('Total Spent and Total Purchases Boxplot')
plt.show()

In [ ]:
df['Total_Purchases'].sort_values(ascending=False).head()

In [ ]:
df['Total_Spent'].sort_values(ascending=False).head()

The flagged outliers in `Total_Spent` and `Total_Purchases` were left untreated. The absolute
deviation from the upper whisker is negligible in both cases, making them a statistical
artefact of the IQR rule rather than genuine anomalies. Removing them without evidence of
measurement error would reduce data integrity without any meaningful benefit to the model.

In [ ]:
products = [col for col in df.columns if 'Mnt' in col]
prod_dict = df[products].sum().to_dict()
global_spent = df['Total_Spent'].sum()
global_avg = df['Total_Spent'].mean()
product_share = round(df[products].sum()*100/global_spent, 2).to_frame().reset_index()
product_share.columns = ['Product', 'Pct']
product_share['Gross'] = product_share['Product'].map(prod_dict)
product_share.sort_values(by='Pct', ascending=False, inplace=True, ignore_index=True)
product_share['Product'] = product_share['Product'].map(lambda x: x.replace('Mnt', '').replace('Products', '').replace('Prods',''))
product_share

In [ ]:
fig = px.bar(product_share, 
             x='Product', y='Pct', 
             hover_data= 'Gross', 
             title='Product Share',
             labels = {'Pct': 'Product Share %'},
             color='Product')
fig.update_yaxes(rangemode="tozero")
fig.show()

## Validating the Highest_Spent Feature

We analyze the Product Share distribution to confirm the statistical viability of a "Dominant Category" feature.
- Diverse Behavioral Signals: The share is not monopolized by a single product; while Wines and Meat lead, there is significant enough variance across other categories to create distinct segments.
- Feature Justification: This variety confirms that customers have unique spending affinities. Creating a categorical Highest_Spent feature allows the XGBoost model to split data based on behavioral personas rather than just raw monetary volume, providing a cleaner signal for predicting campaign responses.

In [ ]:
df['Highest_Spent'] = df[products].idxmax(axis=1)
df['Highest_Spent'] = df['Highest_Spent'].str.replace('Mnt','').str.replace('Products', '').str.replace('Prods', '')
display(df[['Highest_Spent', 'Total_Spent']].head())

In [ ]:
df['Highest_Spent'].value_counts().to_frame()

__<code>df['Dt_Customer']</code> is assigned as <code>object</code>. We wiil convert the column to <code>datetime</code>.__

__<code>df['Year_Birth']</code>, on the other hand, will be left as <code>int</code>. We will use this column to create another one called <code>df['Age']</code> and analyse the age composition of our customers.__

In [ ]:
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], dayfirst = True)
print(df['Dt_Customer'].dtype)

In [ ]:
current_year = int(dt.now().year)
df['Age'] = current_year - df['Year_Birth']
df['Age'].head().to_frame()

In [ ]:
print(f"The maximum age of our customers is {df['Age'].max()} years old, our youngest customer is {df['Age'].min()} years old and the average age is {df['Age'].mean():.2f} and the age median is {df['Age'].median()}")

An initial inspection of the Age distribution reveals values as high as 120+, which are likely erroneous or representative of "placeholder" data. We will utilize a box plot to visualize the extent of these inconsistencies and determine the appropriate threshold for removing unrealistic outliers before modeling.

In [ ]:
max_age = df['Age'].max()
df['Age'].plot(kind='box')
plt.ylabel('Age')
plt.xlabel('Distribution')
plt.annotate(f'Max: {max_age}',
             xy=(1, max_age),
             xytext=(1.1, max_age + 2),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=5))
plt.axhline(max_age, color='red', linestyle='--', alpha=0.4)
plt.title('Outlier Detection: Age Distribution')
plt.ylabel('Age')
plt.show()

__The boxplot shows clear outliers. Let's plot the column in descending order below__

In [ ]:
df['Age'].sort_values(ascending=False).to_frame()

__There are three clear outliers. Let's list them, analise their data regarding Income, and their behavior__

In [ ]:
age_outlier_idxes = list(df.query('Age>=123').index)
print('Age outliter indexes: ', age_outlier_idxes)
display(df.query('Age>=123')) 

In [ ]:
for idx in age_outlier_idxes:
    print('ID', df.loc[idx]['ID'])
    print('Age:', df.loc[idx]['Age'])
    print(f"Total Spent: {df.loc[idx]['Total_Spent']} x Average Spent: {df['Total_Spent'].mean():.2f}")
    print('=='*45)

### Visualizing Age Distribution Without Outliers
To assess the true demographic profile of the customer base, we will visualize the Age distribution after removing the identified extreme outliers (IDs `7829`, `11004`, and `1150`). This allows us to confirm if the remaining data follows a more natural distribution suitable for clustering and modeling.

In [ ]:
age_series = df['Age'].drop(age_outlier_idxes, axis=0)
n = len(age_series)
sturges_bins = int(np.ceil(1 + np.log2(n)))
sns.histplot(age_series, bins=sturges_bins, stat='count', kde=True)
plt.title('Age Distribution')
plt.show()

As observed, removing the extreme outliers has successfully normalized the Age distribution, centering the customer base around a more realistic demographic mean. By neutralizing the skew caused by impossible age values, we have stabilized the feature's variance, ensuring that the column now provides a clean, reliable signal for subsequent clustering and classification tasks.

To maintain the integrity of our clustering and predictive analysis, we are performing a targeted outlier treatment for biologically improbable age values (`>120`).
- Records `192` and `239` (IDs `7829` and `1150`): These will be dropped entirely. Their impossible ages combined with negligible spending suggest corrupted "ghost" data that provides no predictive value.
-  Customer `ID 1150`: We will retain this record but replace the age with the mean of the population. Since this customer is active, this preserves a valuable behavioral signal while neutralizing the statistical skew.
Because this issue affects < 0.1\% of the dataset, mean imputation is a robust choice that saves a perfectly good row of transactional data without significantly altering the feature's variance.

In [ ]:
df.drop([192,239], axis=0, inplace=True)

In [ ]:
age_mean = int(age_series.mean())
print(f"Customers' average age: {age_mean}")

__Let's now verify the new value of Client 1150__

In [ ]:
df.at[339, 'Age'] = age_mean
print(f"Client {df.at[339, 'ID']} new age is now {df.at[339, 'Age']}")

__Let's analyse the column <code>Age</code> and plot a boxplot__

In [ ]:
print(f"The maximum age of our customers is {df['Age'].max()} years old, our youngest customer is {df['Age'].min()} years old and the average age is {df['Age'].mean():.2f} and the age median is {df['Age'].median()}")

In [ ]:
df['Age'].plot(kind='box')
plt.title('Boxplot - Age Distribution')
plt.show()

__Visually, everything is order. Let's just see the values of the youngest customers to make sure that are legitimate observations and not measurement errors, like the previous ones already treated above__

In [ ]:
df['Age'].sort_values(ascending = False).head(20).to_frame()

__And after all this cleaning, one question pops up: Are <code>Age</code> and <code>Income</code> correlated? Let's find out below.__

In [ ]:
sns.regplot(data=df, x = 'Income', y = 'Age' )
plt.title('Age x Income Correlation')
plt.show()

The correlation analysis is currently skewed by an extreme `Income` outlier. This single observation is exerting disproportionate leverage on the regression slope, masking the actual relationship between age and purchasing power for the remaining `99.9%` of the dataset. Furthermore, as identified in our initial data check, `Income` is the only feature containing null values. Given these two distinct issues—statistical noise from an extreme outlier and missing information—we will conduct a deeper investigation into this column to ensure our eventual imputation and cleaning strategies are based on a representative distribution of the customer base.

In [ ]:
df[df['Income'].isnull()]

## `Income` boxplot

In [ ]:
df['Income'].plot(kind='box')
plt.title('Income Distribution')
plt.show()

In [ ]:
income_outlier = df['Income'].max()
income_outlier_record = df.iloc[df['Income']==income_outlier].index
df.drop(income_outlier_record, axis=0, inplace = True)
if income_outlier not in df.index:
    print('Outlier removed')

In [ ]:
plt.figure(figsize=(8,6))
df['Income'].plot(kind='box')
plt.title('Income Distribution')
plt.show()

In [ ]:
n = len(df['Income'].dropna())
sturges_bins = int(np.ceil(1 + np.log2(n)))
sns.histplot(df.dropna(), x='Income', bins = sturges_bins, kde=True)

In [ ]:
print(f"Average Income: {round(df['Income'].mean(),2)}")
print(f"Income Median: {round(df['Income'].median(),2)}")
print(f"Income Standard Deviation: {round(df['Income'].std(),2)}")

Since the removal of extreme outliers has restored a near-normal distribution to the `Income` feature, we will impute missing values using the median. This is a robust choice that preserves the central tendency of the data while remaining resistant to any residual skewness, ensuring the statistical integrity of the feature is maintained for both clustering and classification models.

In [ ]:
income_median = df['Income'].median()
df['Income'] = df['Income'].fillna(round(income_median,2)) 
print(f"Average Income: {round(df['Income'].mean(),2)}")
print(f"Income Median: {round(df['Income'].median(),2)}")
print(f"Income Standard Deviation: {round(df['Income'].std(),2)}")
print(df['Income'].isnull().sum())

In [ ]:
sns.regplot(data=df, x = 'Income', y = 'Age' )
plt.title('Age x Income Correlation')
plt.show()

Even after treating the outliers and imputing missing values, the regression analysis confirms there is no significant linear correlation between Age and Income. The dense cluster of data points across all age groups suggests that in this specific market, purchasing power is not a function of seniority or life stage. This is a critical finding for our upcoming modeling phase; it indicates that these two features provide independent information, and we should look toward other variables—such as marital status, education level or household composition—to explain the variance in customer spending behavior.

__Let's analise another fundamental aspect for customers' behavior: their _Marital Status_. Further on, we will find how the household composition impacts the customers' choices.__ 

__What are the _Marital Statuses_ on our dataset?__

In [ ]:
for i in df['Marital_Status'].unique():
    print(i)

__<code>Alone</code>, <code>Absurd</code> and <code>YOLO</code> are not exactly conventional _Marital Statuses_. Let's now analise the absolute and relative frequency of these categorical values__

In [ ]:
frequency= df['Marital_Status'].value_counts()
percentages = round((df['Marital_Status'].value_counts(normalize=True) * 100),2)
freq_marital = pd.DataFrame({'Frequency':frequency, 'Percentages (%)': percentages})
freq_marital

__Now, let's see the absolute and relative frequency of <code>Education</code>__

In [ ]:
frequency_education= df['Education'].value_counts()
percentages_education = round((df['Education'].value_counts(normalize=True) * 100),2)
freq_education = pd.DataFrame({'Frequency':frequency_education, 'Percentages (%)': percentages_education})
freq_education.index.name='Education'
freq_education

__Now, let's cross-tabulate <code>Education</code> and <code>Marital_Status</code>. We will create two tables. One with the absolute frequency and another one with the relative frequency.__

In [ ]:
# Absolute frequency
ct = pd.crosstab(df['Education'], df['Marital_Status'])
# Relative frequency (percentage of total)
ct_pct = round(pd.crosstab(df['Education'], df['Marital_Status'], normalize='all') * 100,2) 

In [ ]:
print('Relative frequency')
ct_pct

In [ ]:
print('Absolute Frequency')
ct

Cross-tabulating `Education` and `Marital_Status` reveals that non-standard entries like `Absurd` and  `YOLO` are statistically insignificant noise, appearing in less than 0.1% of the population. To streamline our feature engineering and prevent model overfitting on these rare categories, we will consolidate `Alone`, `Absurd`, and `YOLO`into the `Single` category.

In [ ]:
df['Marital_Status'] = df['Marital_Status'].replace(['Alone', 'Absurd', 'YOLO'], 'Single')
print(df['Marital_Status'].value_counts())

In [ ]:
ct = pd.crosstab(df['Education'], df['Marital_Status'])
ct_pct = round(pd.crosstab(df['Education'], df['Marital_Status'], normalize='all') * 100,2) 

In [ ]:
import plotly.graph_objects as go

# 1. Prepare data (ensure columns are the marital statuses)
# ct_pct should have Education as index and Marital Statuses as columns
categories = ct_pct.index.tolist()  # Education levels
status_cols = ct_pct.columns.tolist()  # Marital Statuses

fig = go.Figure()

# 2. Add a trace for each Education level (initially only the first one is visible)
for i, edu in enumerate(categories):
    fig.add_trace(
        go.Bar(
            x=status_cols,
            y=ct_pct.loc[edu],
            name=edu,
            visible=(i == 0), # Only first education level shown by default
            text=[f"{v:.2f}%" for v in ct_pct.loc[edu]],
            textposition='auto',
        )
    )

# 3. Create the dropdown menu
dropdown_buttons = []
for i, edu in enumerate(categories):
    # Create a list of booleans: True for the current trace, False for others
    visibility = [False] * len(categories)
    visibility[i] = True
    
    dropdown_buttons.append(dict(
        label=edu,
        method="update",
        args=[{"visible": visibility},
              {"title": f"Marital Status Breakdown for {edu}"}]
    ))

# 4. Update layout with the menu
fig.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=dropdown_buttons,
        direction="down",
        pad={"r": 10, "t": 10},
        showactive=True,
        x=0.1,
        xanchor="left",
        y=1.15,
        yanchor="top"
    )],
    title=f"Marital Status Breakdown for {categories[0]}",
    yaxis_title="Percentage (%)",
    template="plotly_white",
    height=500
)

fig.show()

We will now create an aggregated DataFrame grouped by `Marital_Status` and `Education`,
computing the mean of all relevant features. This summary table, combined with the cleaned
dataset and a equivalent one using standard deviations, will be used to synthetically
extrapolate data to hundreds of thousands of rows. The resulting dataset will then be
uploaded to BigQuery, where the trained model will generate predictions at scale.

In [ ]:
marital_status_education_df = df.groupby(['Marital_Status','Education']).agg({
    'Income':'mean',
    'Kidhome':'mean',
    'Teenhome':'mean',
    'Recency':'mean',
    'Complain':'mean',
    'MntWines':'mean',
    'MntFruits':'mean',
    'MntMeatProducts':'mean',
    'MntFishProducts':'mean',
    'MntSweetProducts':'mean',
    'MntGoldProds':'mean',
    'NumDealsPurchases': 'mean',
    'NumWebPurchases': 'mean',
    'NumCatalogPurchases':'mean',
    'NumStorePurchases':'mean',
    'NumWebVisitsMonth':'mean',
    'AcceptedCmp3':'mean',
    'AcceptedCmp4': 'mean',
    'AcceptedCmp5': 'mean',
    'AcceptedCmp1': 'mean',
    'AcceptedCmp2': 'mean',
    'Response': 'mean',
    'Total_Spent': 'mean',
    'Age': 'mean', 
    'Total_Purchases': 'mean'
}
).reset_index()

After aggregating by group means, float columns are split into two groups: those representing
counts (`Kidhome`, `Teenhome`, `Total_Purchases`, etc.) are rounded to the nearest integer
using ceiling/floor logic, since fractional values are not meaningful for these features;
while monetary columns (`Income`, `Total_Spent`) and product amounts are left as floats, as
their decimal precision carries informational value.

In [ ]:
all_floats = marital_status_education_df.select_dtypes(include='float').columns.to_list()
to_int = marital_status_education_df.select_dtypes(include='float').columns.drop(['Income', 'Total_Spent']+products).to_list()
to_round = [i for i in all_floats if i not in to_int]
print('To int:', to_int)
print('To round:', to_round)

In [ ]:
int_transform = marital_status_education_df[to_int]
ceils = np.ceil(int_transform)
floors = np.floor(int_transform)
transformed = np.where(ceils - int_transform <= 0.5, ceils, floors)
marital_status_education_df[to_int] = transformed.astype('int64')
marital_status_education_df[to_int].head()

In [ ]:
marital_status_education_df[to_round] = marital_status_education_df[to_round].round(2)
marital_status_education_df[to_round].head()

In [ ]:
marital_status_education_df

In [ ]:
marital_status_education_df_std = df.groupby(['Marital_Status','Education']).agg({
    'Income':'std',
    'Kidhome':'std',
    'Teenhome':'std',
    'Recency':'std',
    'Complain':'std',
    'MntWines':'std',
    'MntFruits':'std',
    'MntMeatProducts':'std',
    'MntFishProducts':'std',
    'MntSweetProducts':'std',
    'MntGoldProds':'std',
    'NumDealsPurchases': 'std',
    'NumWebPurchases': 'std',
    'NumCatalogPurchases':'std',
    'NumStorePurchases':'std',
    'NumWebVisitsMonth':'std',
    'AcceptedCmp3':'std',
    'AcceptedCmp4': 'std',
    'AcceptedCmp5': 'std',
    'AcceptedCmp1': 'std',
    'AcceptedCmp2': 'std',
    'Response': 'std',
    'Total_Spent': 'std',
    'Age': 'std', 
    'Total_Purchases': 'std'
}
).reset_index()

In [ ]:
all_floats = marital_status_education_df_std.select_dtypes(include='float').columns.to_list()
to_int = marital_status_education_df_std.select_dtypes(include='float').columns.drop(['Income', 'Total_Spent']+products).to_list()
to_round = [i for i in all_floats if i not in to_int]
print('To int:', to_int)
print('To round:', to_round)

In [ ]:
int_transform = marital_status_education_df_std[to_int]
ceils = np.ceil(int_transform)
floors = np.floor(int_transform)
transformed = np.where(ceils - int_transform <= 0.5, ceils, floors)
marital_status_education_df_std[to_int] = transformed.astype('int64')
marital_status_education_df_std[to_int].head()

In [ ]:
marital_status_education_df_std[to_round] = marital_status_education_df_std[to_round].round(2)
marital_status_education_df_std[to_round].head()

In [ ]:
marital_status_education_df_std

We also extract the proportional distribution of `Highest_Spent` — the product category
where each customer spends the most — grouped by `Marital_Status` and `Education`. These
proportions will be used when generating synthetic entries, ensuring that the category
distribution within each segment is preserved and that the expanded dataset remains
faithful to the spending patterns observed in the original data.

In [ ]:
df_highest_spent = df.groupby(['Marital_Status', 'Education'])['Highest_Spent'].value_counts(normalize=True).to_frame().reset_index()
df_highest_spent.head()